# 🛡️ Notebook 04 — Sanitizer Agent (HIPAA Compliance)
### Multi-Agent System for Secure Clinical Summarization

## What This Notebook Does
Builds the Sanitizer Agent using Microsoft Presidio to detect and remove
Protected Health Information (PHI) from generated summaries.

PHI types detected:
- Person names (patient, provider, family)
- Phone numbers
- Email addresses
- Dates (partial masking to preserve clinical context)
- Physical addresses
- Social Security Numbers
- Medical Record Numbers (custom: MRN + 6-10 digits)
- Account Numbers (custom: Account + 6-12 digits)
- Patient IDs (custom: multiple formats)

Replacement strategy (preserves readability):
- Names: `[PATIENT]`, `[PROVIDER]`, `[FAMILY]`
- Dates: `12/15/2024` → `12/**/2024`
- Medical IDs: `[ID]`
- Phone/Email/Address: `[PHONE]`, `[EMAIL]`, `[LOCATION]`

**Runtime: CPU only**


In [ ]:
# ── Step 1: Mount Drive and set paths ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR   = '/content/drive/MyDrive/clinical_mas'
AGENTS_DIR = f'{BASE_DIR}/agents'
os.makedirs(AGENTS_DIR, exist_ok=True)

AGENT_PATH = f'{AGENTS_DIR}/sanitizer_agent.py'
print('Drive mounted')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [ ]:
# ── Step 2: Install Microsoft Presidio ────────────────────────────────────────
# presidio-analyzer  : detects PHI entities using NER + patterns
# presidio-anonymizer: replaces detected PHI with placeholders
# spacy + en_core_web_lg: NLP model for named entity recognition

!pip install -q presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg -q
print('Presidio installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Presidio installed


In [ ]:
# ── Step 3: Imports ───────────────────────────────────────────────────────────
import re
import time
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
print('Imports OK')


Imports OK


In [ ]:
# ── Step 4: Build custom PHI recognizers ──────────────────────────────────────
# Presidio has built-in recognizers for common PII (names, phones, SSN, etc.)
# We add custom recognizers for healthcare-specific identifiers.

# Medical Record Number: MRN followed by 6-10 digits
mrn_recognizer = PatternRecognizer(
    supported_entity='MEDICAL_RECORD_NUMBER',
    patterns=[
        Pattern('MRN_FORMAT',    r'\bMRN[:\s#]+\d{6,10}\b', 0.9),
        Pattern('MRN_FORMAT_2',  r'\bM\.R\.N[:\s#]+\d{6,10}\b', 0.9),
        Pattern('MR_NUMBER',     r'\bMR[:\s#]+\d{6,10}\b', 0.85),
    ]
)

# Account number: Account followed by 6-12 digits
account_recognizer = PatternRecognizer(
    supported_entity='ACCOUNT_NUMBER',
    patterns=[
        Pattern('ACCOUNT_FORMAT', r'\b[Aa]ccount[:\s#]+\d{6,12}\b', 0.9),
        Pattern('ACCT_FORMAT',    r'\b[Aa]cct[:\s#]+\d{6,12}\b', 0.85),
    ]
)

# Patient ID: various formats
patient_id_recognizer = PatternRecognizer(
    supported_entity='PATIENT_ID',
    patterns=[
        Pattern('PID_FORMAT',     r'\b[Pp]atient\s+ID[:\s#]+[\w\-]+\b', 0.9),
        Pattern('PT_FORMAT',      r'\b[Pp]t[:\s#]+ID[:\s#]+[\w\-]+\b', 0.85),
        Pattern('PATIENT_NUM',    r'\b[Pp]atient\s+[Nn]umber[:\s#]+\d{4,12}\b', 0.9),
    ]
)

print('Custom recognizers defined')
print('  - Medical Record Number (MRN + 6-10 digits)')
print('  - Account Number (Account + 6-12 digits)')
print('  - Patient ID (multiple formats)')


Custom recognizers defined
  - Medical Record Number (MRN + 6-10 digits)
  - Account Number (Account + 6-12 digits)
  - Patient ID (multiple formats)


In [ ]:
# ── Step 5: Build the SanitizerAgent class ────────────────────────────────────

class SanitizerAgent:
    '''
    Sanitizer Agent - HIPAA PHI detection and removal.

    Uses Microsoft Presidio with custom healthcare recognizers.
    Confidence thresholds:
      >= 0.8 : auto-anonymize
      0.5-0.8: anonymize (configurable to flag for review instead)
    '''

    # Map Presidio entity types to replacement labels
    REPLACEMENTS = {
        'PERSON'                : '[PATIENT]',
        'PHONE_NUMBER'          : '[PHONE]',
        'EMAIL_ADDRESS'         : '[EMAIL]',
        'LOCATION'              : '[LOCATION]',
        'US_SSN'                : '[SSN]',
        'US_DRIVER_LICENSE'     : '[ID]',
        'MEDICAL_RECORD_NUMBER' : '[ID]',
        'ACCOUNT_NUMBER'        : '[ID]',
        'PATIENT_ID'            : '[ID]',
        'DATE_TIME'             : None,  # handled separately (partial masking)
        'URL'                   : '[URL]',
        'IP_ADDRESS'            : '[IP]',
    }

    # Date partial masking: keep month/year, mask day
    DATE_PATTERNS = [
        (re.compile(r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b'), r'\1/**/\3'),   # MM/DD/YYYY -> MM/**/YYYY
        (re.compile(r'\b(\d{4})-(\d{2})-(\d{2})\b'),     r'\1-\2-**'),    # YYYY-MM-DD -> YYYY-MM-**
        (re.compile(r'\b(January|February|March|April|May|June|July|August|'
                    r'September|October|November|December)\s+\d{1,2},?\s+(\d{4})\b',
                    re.IGNORECASE),                                         r'\1 **, \2'),
    ]

    def __init__(self, min_confidence=0.5):
        self.min_confidence = min_confidence

        # Initialize NLP engine with spacy large model
        provider = NlpEngineProvider(nlp_configuration={
            'nlp_engine_name': 'spacy',
            'models': [{'lang_code': 'en', 'model_name': 'en_core_web_lg'}]
        })
        nlp_engine = provider.create_engine()

        self.analyzer  = AnalyzerEngine(nlp_engine=nlp_engine)
        self.anonymizer = AnonymizerEngine()

        # Register custom recognizers
        self.analyzer.registry.add_recognizer(mrn_recognizer)
        self.analyzer.registry.add_recognizer(account_recognizer)
        self.analyzer.registry.add_recognizer(patient_id_recognizer)

        print(f'SanitizerAgent ready (min_confidence={min_confidence})')

    def _mask_dates(self, text):
        '''Apply partial masking to dates: 12/15/2024 -> 12/**/2024'''
        for pattern, replacement in self.DATE_PATTERNS:
            text = pattern.sub(replacement, text)
        return text

    def sanitize(self, text):
        '''
        Detect and remove PHI from text.

        Args:
            text: generated summary to sanitize

        Returns:
            dict with sanitized_text, entities_found, and latency
        '''
        t0 = time.perf_counter()

        # Step 1: Apply date partial masking
        text_after_dates = self._mask_dates(text)

        # Step 2: Run Presidio analyzer
        results = self.analyzer.analyze(
            text=text_after_dates,
            language='en',
            score_threshold=self.min_confidence,
            entities=[
                'PERSON', 'PHONE_NUMBER', 'EMAIL_ADDRESS', 'LOCATION',
                'US_SSN', 'US_DRIVER_LICENSE',
                'MEDICAL_RECORD_NUMBER', 'ACCOUNT_NUMBER', 'PATIENT_ID',
                'URL', 'IP_ADDRESS',
            ]
        )

        # Step 3: Build operator config for each entity type
        operators = {}
        for entity_type, replacement in self.REPLACEMENTS.items():
            if replacement and entity_type != 'DATE_TIME':
                operators[entity_type] = OperatorConfig('replace', {'new_value': replacement})

        # Step 4: Apply anonymization
        if results:
            anonymized = self.anonymizer.anonymize(
                text=text_after_dates,
                analyzer_results=results,
                operators=operators,
            )
            sanitized_text = anonymized.text
        else:
            sanitized_text = text_after_dates

        latency_ms = round((time.perf_counter() - t0) * 1000, 2)

        entities_found = [
            {'type': r.entity_type, 'score': round(r.score, 3),
             'text': text_after_dates[r.start:r.end]}
            for r in results
        ]

        return {
            'sanitized_text': sanitized_text,
            'original_text' : text,
            'entities_found': entities_found,
            'n_entities'    : len(entities_found),
            'latency_ms'    : latency_ms,
            'phi_detected'  : len(entities_found) > 0,
        }


sanitizer = SanitizerAgent(min_confidence=0.5)


SanitizerAgent ready (min_confidence=0.5)


In [ ]:
# ── Step 6: Test on sample texts ──────────────────────────────────────────────

test_samples = [
    # Clean text - should pass unchanged
    'You were admitted for congestive heart failure. Your condition improved with treatment. '
    'Take furosemide once daily. Follow up with cardiology in 2 weeks.',

    # Contains person name
    'Patient John Smith was admitted on 12/15/2024 for pneumonia. '
    'Dr. Jane Doe prescribed amoxicillin. Call 555-123-4567 for follow-up.',

    # Contains MRN, account number, address
    'MRN: 1234567 Account: 98765432 '
    'Patient ID: PT-001234 '
    'Address: 123 Main Street, Boston, MA 02101 '
    'Email: patient@email.com '
    'SSN: 123-45-6789',

    # Date masking test
    'Admitted January 5, 2024. Discharged on 01/15/2024. '
    'Follow-up scheduled for 2024-02-01.',
]

print('=== Sanitizer Test ===')
for i, sample in enumerate(test_samples, 1):
    result = sanitizer.sanitize(sample)
    print(f'\nTest {i}:')
    print(f'  Input    : "{sample[:100]}"')
    print(f'  Output   : "{result["sanitized_text"][:100]}"')
    print(f'  Entities : {result["n_entities"]} found: '
          f'{[e["type"] for e in result["entities_found"]]}')
    print(f'  Latency  : {result["latency_ms"]} ms')


=== Sanitizer Test ===



Test 1:
  Input    : "You were admitted for congestive heart failure. Your condition improved with treatment. Take furosem"
  Output   : "You were admitted for congestive heart failure. Your condition improved with treatment. Take furosem"
  Entities : 0 found: []
  Latency  : 249.76 ms

Test 2:
  Input    : "Patient John Smith was admitted on 12/15/2024 for pneumonia. Dr. Jane Doe prescribed amoxicillin. Ca"
  Output   : "Patient [PATIENT] was admitted on 12/**/2024 for pneumonia. Dr. [PATIENT] prescribed amoxicillin. Ca"
  Entities : 2 found: ['PERSON', 'PERSON']
  Latency  : 37.81 ms

Test 3:
  Input    : "MRN: 1234567 Account: 98765432 Patient ID: PT-001234 Address: 123 Main Street, Boston, MA 02101 Emai"
  Output   : "[ID] [ID] [ID] Address: 123 Main Street, [LOCATION], MA 02101 Email: [EMAIL] SSN: 123-45-6789"
  Entities : 6 found: ['EMAIL_ADDRESS', 'MEDICAL_RECORD_NUMBER', 'ACCOUNT_NUMBER', 'PATIENT_ID', 'LOCATION', 'URL']
  Latency  : 51.62 ms

Test 4:
  Input    : "Admitted J

In [ ]:
# ── Step 7: Save sanitizer_agent.py to Drive ──────────────────────────────────

code = '''"""Sanitizer Agent - HIPAA PHI detection and removal using Microsoft Presidio.
Detects names, phones, emails, dates, SSNs, MRNs, account numbers, patient IDs.
Applies category-specific replacements and partial date masking.
"""
import re, time
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

mrn_recognizer = PatternRecognizer(
    supported_entity="MEDICAL_RECORD_NUMBER",
    patterns=[
        Pattern("MRN",  r"\\bMRN[:\\s#]+\\d{6,10}\\b", 0.9),
        Pattern("MR",   r"\\bMR[:\\s#]+\\d{6,10}\\b", 0.85),
    ]
)
account_recognizer = PatternRecognizer(
    supported_entity="ACCOUNT_NUMBER",
    patterns=[Pattern("ACCT", r"\\b[Aa]ccount[:\\s#]+\\d{6,12}\\b", 0.9)]
)
patient_id_recognizer = PatternRecognizer(
    supported_entity="PATIENT_ID",
    patterns=[Pattern("PID", r"\\b[Pp]atient\\s+ID[:\\s#]+[\\w\\-]+\\b", 0.9)]
)

class SanitizerAgent:
    REPLACEMENTS = {
        "PERSON": "[PATIENT]", "PHONE_NUMBER": "[PHONE]", "EMAIL_ADDRESS": "[EMAIL]",
        "LOCATION": "[LOCATION]", "US_SSN": "[SSN]", "US_DRIVER_LICENSE": "[ID]",
        "MEDICAL_RECORD_NUMBER": "[ID]", "ACCOUNT_NUMBER": "[ID]", "PATIENT_ID": "[ID]",
    }
    DATE_PATTERNS = [
        (re.compile(r"\\b(\\d{1,2})/(\\d{1,2})/(\\d{4})\\b"), r"\\1/**/\\3"),
        (re.compile(r"\\b(\\d{4})-(\\d{2})-(\\d{2})\\b"),     r"\\1-\\2-**"),
    ]

    def __init__(self, min_confidence=0.5):
        provider = NlpEngineProvider(nlp_configuration={
            "nlp_engine_name": "spacy",
            "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}]
        })
        self.analyzer   = AnalyzerEngine(nlp_engine=provider.create_engine())
        self.anonymizer = AnonymizerEngine()
        for rec in [mrn_recognizer, account_recognizer, patient_id_recognizer]:
            self.analyzer.registry.add_recognizer(rec)
        self.min_confidence = min_confidence

    def _mask_dates(self, text):
        for pat, rep in self.DATE_PATTERNS:
            text = pat.sub(rep, text)
        return text

    def sanitize(self, text):
        t0 = time.perf_counter()
        text2 = self._mask_dates(text)
        results = self.analyzer.analyze(text=text2, language="en",
                                        score_threshold=self.min_confidence)
        ops = {k: OperatorConfig("replace", {"new_value": v})
               for k, v in self.REPLACEMENTS.items()}
        sanitized = (self.anonymizer.anonymize(text=text2, analyzer_results=results,
                                               operators=ops).text
                     if results else text2)
        return {
            "sanitized_text": sanitized, "original_text": text,
            "entities_found": [{"type": r.entity_type, "score": round(r.score, 3)}
                               for r in results],
            "n_entities": len(results),
            "latency_ms": round((time.perf_counter()-t0)*1000, 2),
            "phi_detected": len(results) > 0,
        }
'''

with open(AGENT_PATH, 'w') as f:
    f.write(code)
print(f'Saved sanitizer_agent.py ({os.path.getsize(AGENT_PATH):,} bytes)')


Saved sanitizer_agent.py (3,086 bytes)


In [ ]:
# ── Step 8: Final Validation ─────────────────────────────────────────────────

# Test PHI detection on known PHI
test_phi = 'Patient John Smith (MRN: 1234567) was admitted on 01/15/2024. Contact: john@email.com'
result = sanitizer.sanitize(test_phi)

phi_detected  = result['n_entities'] > 0
date_masked   = '**/2024' in result['sanitized_text'] or '/**/2024' in result['sanitized_text']
clean_passes  = sanitizer.sanitize(
    'You were admitted for heart failure. Take furosemide daily.'
)['sanitized_text'] == 'You were admitted for heart failure. Take furosemide daily.'

print('=' * 55)
print('  SANITIZER AGENT - FINAL VALIDATION')
print('=' * 55)
checks = [
    ('PHI detected in sample',  phi_detected,                  f'{result["n_entities"]} entities'),
    ('Date partially masked',   date_masked,                   'MM/**/YYYY format'),
    ('Clean text passes through', clean_passes,                'no false positives'),
    ('sanitizer_agent.py saved', os.path.exists(AGENT_PATH),   'agents/sanitizer_agent.py'),
]
all_ok = True
for name, passed, detail in checks:
    icon = 'PASS' if passed else 'FAIL'
    if not passed: all_ok = False
    print(f'  [{icon}] {name:<35} {detail}')
print('=' * 55)
if all_ok:
    print('  ALL CHECKS PASSED - Sanitizer Agent is ready!')
    print('  Next: Run 05_orchestrator.ipynb')
else:
    print('  Some checks failed - see above')
print('=' * 55)


  SANITIZER AGENT - FINAL VALIDATION
  [PASS] PHI detected in sample              4 entities
  [PASS] Date partially masked               MM/**/YYYY format
  [PASS] Clean text passes through           no false positives
  [PASS] sanitizer_agent.py saved            agents/sanitizer_agent.py
  ALL CHECKS PASSED - Sanitizer Agent is ready!
  Next: Run 05_orchestrator.ipynb
